# MISP-Bench: Item-Level Quality Audit

Reproduces the per-question exclusion flags described in §3.1 (Table 1) and
`EXCLUSIONS.md`. Six categories cover 770 items (31% of the 2,494-item pool):

| Category | n | Detection | Domain |
|---|---:|---|---|
| `choice_type_multi` | 732 | MedMCQA `choice_type` field | medical |
| `image_referencing` | 28 | keyword filter + manual review | medical |
| `exact_duplicate_options` | 12 | byte-equal option text | medical |
| `math_dist_eq_correct` | 6 | `\|distractor − gold\| < 0.5` | math |
| `label_error` | 2 | unanimous-wrong + textual contradiction | medical |
| `wr_leaks_correct` | 1 | gold appears in `wrong_reasoning` | medical |
| **Net union** | **770** | | |

This notebook produces a single artifact:
`tables/t0_question_flags.csv` — the per-question flag table consumed by
`04_analysis.ipynb`. **Run this before the analysis notebook.**

The four hand-curated lists (`EXACT_DUP_EXCLUDED_IDS`, `IMAGE_REFERENCING_IDS`,
`LABEL_ERROR_IDS`) are reproduced verbatim with provenance comments.
The remaining three flags (`choice_type_multi`, `math_dist_eq_correct`,
`wr_leaks_correct`) are computed automatically from the JSON.


In [ ]:
import json, re
from pathlib import Path
import pandas as pd

QUESTION_SET_PATH = "Benchmark.json"
OUT_DIR = Path("tables"); OUT_DIR.mkdir(exist_ok=True)

with open(QUESTION_SET_PATH, "r", encoding="utf-8") as f:
    qset = json.load(f)
print(f"Loaded {len(qset['questions']):,} questions "
      f"(medical={qset['meta']['n_medical']}, math={qset['meta']['n_math']})")


## 1. Hand-curated exclusion lists

These three lists were produced by manual review (§3.1, S2). They are
reproduced verbatim so the audit is byte-deterministic from the JSON alone.


In [ ]:
# ────────────────────────────────────────────────────────────────────
# (1) EXACT_DUP — 12 items where two or more options share byte-identical text
#     Detected automatically (option set comparison) and confirmed by inspection.
# ────────────────────────────────────────────────────────────────────
EXACT_DUP_EXCLUDED_IDS = frozenset({
    "med_00318", "med_00988", "med_01341", "med_01406",
    "med_01632", "med_01854", "med_02409", "med_02733",
    "med_03484", "med_03651", "med_03750", "med_03844",
})

# ────────────────────────────────────────────────────────────────────
# (2) IMAGE_REFERENCING — 28 items requiring visual input
#     51 keyword-filtered candidates → 2-author manual review →
#     3 tiers (explicit / implicit / borderline). 23 false positives rejected.
#     See EXCLUSIONS.md §2 for the full audit protocol.
# ────────────────────────────────────────────────────────────────────
IMAGE_REFERENCING_IDS = frozenset({
    # Tier 1 — explicit visual reference (n=19 confirmed)
    "med_01577", "med_02605", "med_01384", "med_01522",
    "med_04031", "med_02258", "med_01879", "med_03690",
    "med_04059", "med_00810", "med_01030", "med_03163",
    "med_03865", "med_01915", "med_03879", "med_00647",
    "med_00696", "med_02372", "med_03938",
    # Tier 2 — implicit visual dependence (n=4)
    "med_02142", "med_02151", "med_03469", "med_03738",
    # Tier 3 — context-dependent without antecedent (n=5)
    "med_02239", "med_00431", "med_04010", "med_00061",
    "med_02586",
})

# ────────────────────────────────────────────────────────────────────
# (3) LABEL_ERROR — 2 confirmed gold-label errors via dual evidence
#     (a) all inference-pool models pick the same NON-gold option at L1, AND
#     (b) MedMCQA's own explanation describes that non-gold option.
#     Expert clinical judgment is NOT invoked. See EXCLUSIONS.md §3.
# ────────────────────────────────────────────────────────────────────
LABEL_ERROR_IDS = frozenset({
    "med_02165",  # Physiology, Phosphocreatine: gold=A, explanation describes D
    "med_03953",  # SPM, Haddon matrix:        gold=B, explanation describes A
})

print(f"EXACT_DUP_EXCLUDED_IDS:  {len(EXACT_DUP_EXCLUDED_IDS)} items")
print(f"IMAGE_REFERENCING_IDS:   {len(IMAGE_REFERENCING_IDS)} items")
print(f"LABEL_ERROR_IDS:         {len(LABEL_ERROR_IDS)} items")


## 2. Compute per-question flag table

For each question we compute every flag column. A flag set to `True` means
the item triggers that exclusion criterion.

**Global exclusion flags** (drop the item from the main corpus):
- `exact_duplicate_options`
- `image_referencing`
- `label_error`
- `math_dist_eq_correct`
- `wr_leaks_correct`
- `choice_type_multi`

**Sub-analysis flags** (drop only for specific levels):
- `scope_identical` — drops L7 only
- `confident_missing_distractor` — drops L5 only
- `l3_leak` — drops L3 only
- `l4c_padded` — sensitivity analysis on L4c only
- `explanation_short` — sensitivity (≤100 char explanation)


In [ ]:
def compute_question_flags(qset: dict) -> pd.DataFrame:
    """Return one row per question with all audit flags."""
    PAD = "Consider the underlying mechanisms carefully"
    rows = []

    for q in qset["questions"]:
        qid  = q["id"]
        dom  = q["domain"]
        dist = q.get("distractor_answer", "")
        cor  = q.get("correct_answer", "")
        flags = {}

        # ── (a) MATH: distractor numerically equals gold ──
        if dom == "math":
            try:
                if abs(float(str(dist).replace(",", "")) -
                       float(str(cor).replace(",", ""))) < 0.5:
                    flags["math_dist_eq_correct"] = True
            except (ValueError, TypeError):
                pass

        # ── (b) MEDICAL: wrong_reasoning leaks gold letter ──
        if dom == "medical" and cor:
            wr = q.get("wrong_reasoning", "") or ""
            if re.search(rf"\({cor}\)|answer is {cor}\b|option {cor}\b", wr, re.I):
                flags["wr_leaks_correct"] = True

        # ── (c) scope_in == scope_out (L7-only flag) ──
        if q.get("scope_in") and q.get("scope_out"):
            if q["scope_in"].strip() == q["scope_out"].strip():
                flags["scope_identical"] = True

        # ── (d) confident_assertion missing distractor letter (L5-only flag) ──
        if dom == "medical":
            ca = q.get("confident_assertion", "") or ""
            if dist and f"({dist})" not in ca:
                flags["confident_missing_distractor"] = True

        # ── (e) L3 leak: gold appears in correct-hint prefix ──
        l3 = q.get("prompts", {}).get("L3", "")
        hint = l3.split("\n\n")[0] if isinstance(l3, str) else ""
        if dom == "medical" and cor:
            if re.search(rf"\({cor}\)|\banswer is {cor}\b|\boption {cor}\b",
                         hint, re.I):
                flags["l3_leak"] = True
        elif dom == "math" and str(cor).strip():
            if re.search(rf"\b{re.escape(str(cor).strip())}\b", hint):
                flags["l3_leak"] = True

        # ── (f) L4c padding ──
        l4c = q.get("prompts", {}).get("L4c", "")
        if isinstance(l4c, str) and PAD in l4c:
            flags["l4c_padded"] = True

        # ── (g) Hand-curated MedMCQA upstream defects ──
        if qid in EXACT_DUP_EXCLUDED_IDS:
            flags["exact_duplicate_options"] = True
        if qid in IMAGE_REFERENCING_IDS:
            flags["image_referencing"] = True
        if qid in LABEL_ERROR_IDS:
            flags["label_error"] = True

        # ── (h) choice_type=='multi' (post-hoc structural filter) ──
        if dom == "medical":
            ct = q.get("choice_type", None)
            if ct == "multi":
                flags["choice_type_multi"] = True
            elif ct is None:
                flags["choice_type_unknown"] = True   # defensive

        # ── (i) Short-explanation sensitivity (≤100 chars) ──
        if dom == "medical":
            exp_text = q.get("explanation", "") or ""
            if len(exp_text) <= 100:
                flags["explanation_short"] = True

        rows.append({
            "question_id":       qid,
            "domain":            dom,
            "subject":           q.get("subject", "Unknown"),
            "topic":             q.get("topic", "Unknown"),
            "correct_answer":    cor,
            "distractor_answer": dist,
            "distractor_source": q.get("distractor_source", "unknown"),
            "difficulty_length": q.get("difficulty", "unknown"),
            "question_len":      q.get("difficulty_proxy", len(q.get("question", ""))),
            **{k: True for k in flags},
        })

    fdf = pd.DataFrame(rows).fillna(False)

    # Ensure every flag column exists even if 0 items triggered it
    for c in ["math_dist_eq_correct", "wr_leaks_correct", "scope_identical",
              "confident_missing_distractor", "l3_leak", "l4c_padded",
              "exact_duplicate_options", "image_referencing", "label_error",
              "choice_type_multi", "choice_type_unknown", "explanation_short"]:
        if c not in fdf.columns:
            fdf[c] = False

    return fdf


flags_df = compute_question_flags(qset)
print(f"Computed flags for {len(flags_df):,} questions.")


## 3. Reproduce paper Table 1

The next cell reproduces the per-category counts and the net-union exclusion
size. These numbers should match §3.1 Table 1 exactly.


In [ ]:
GLOBAL_EXCLUSION_FLAGS = [
    "choice_type_multi",
    "image_referencing",
    "exact_duplicate_options",
    "math_dist_eq_correct",
    "label_error",
    "wr_leaks_correct",
]

print("┌─────────────────────────────────────────────────────────────┐")
print("│        Six-category dataset audit (paper Table 1)           │")
print("├─────────────────────────────────────────────────────────────┤")
total = 0
for f in GLOBAL_EXCLUSION_FLAGS:
    n = int(flags_df[f].sum())
    total += n
    print(f"│  {f:<30s}  {n:>5,}                       │")
print("├─────────────────────────────────────────────────────────────┤")

excl_mask = flags_df[GLOBAL_EXCLUSION_FLAGS].any(axis=1)
n_union = int(excl_mask.sum())
print(f"│  Sum of categories            {total:>5,}                       │")
print(f"│  Net union (after overlap)    {n_union:>5,}                       │")
print("└─────────────────────────────────────────────────────────────┘")

n_total = len(flags_df)
n_final = n_total - n_union
print(f"\nInitial pool : {n_total:,}")
print(f"Excluded     : {n_union:,}  ({100*n_union/n_total:.1f}%)")
print(f"Final corpus : {n_final:,}")


## 4. Save `tables/t0_question_flags.csv`

This is the artifact `04_analysis.ipynb` reads at the top of its loader.


In [ ]:
out_csv = OUT_DIR / "t0_question_flags.csv"
flags_df.to_csv(out_csv, index=False)

print(f"Wrote: {out_csv}  ({out_csv.stat().st_size/1024:.1f} KB)")
print(f"Rows: {len(flags_df):,}")
print(f"\nFlag column counts:")
for c in flags_df.columns:
    if flags_df[c].dtype == bool:
        n = int(flags_df[c].sum())
        if n > 0:
            print(f"  {c:<32s} {n:>5,}")


## 5. Audit reproducibility check

Sanity-checks that the three hand-curated ID lists actually map to items in
the JSON (no orphan IDs from a stale list).


In [ ]:
all_ids = {q["id"] for q in qset["questions"]}

for name, ids in [
    ("EXACT_DUP_EXCLUDED_IDS", EXACT_DUP_EXCLUDED_IDS),
    ("IMAGE_REFERENCING_IDS",  IMAGE_REFERENCING_IDS),
    ("LABEL_ERROR_IDS",        LABEL_ERROR_IDS),
]:
    missing = ids - all_ids
    status = "OK" if not missing else f"MISSING {len(missing)}: {sorted(missing)}"
    print(f"  [{status:<5}] {name:<26s} {len(ids):>3} items")

# Also verify the auto-detected categories
auto_dup = set()
for q in qset["questions"]:
    if q["domain"] == "medical" and q.get("options"):
        opts = list(q["options"].values())
        if len(set(opts)) < len(opts):
            auto_dup.add(q["id"])

mismatch_dup = auto_dup ^ EXACT_DUP_EXCLUDED_IDS
if mismatch_dup:
    print(f"\n[WARN] EXACT_DUP autodetect disagrees with hand list: "
          f"only-auto={auto_dup-EXACT_DUP_EXCLUDED_IDS}, "
          f"only-hand={EXACT_DUP_EXCLUDED_IDS-auto_dup}")
else:
    print(f"\n[OK] EXACT_DUP autodetect ({len(auto_dup)}) == hand list. "
          f"The hand list is reproducible from the JSON alone.")
